In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv("../data/cancer-risk-factors.csv")

In [4]:
df.head(5).T

,0,1,2,3,4
Patient_ID,LU0000,LU0001,LU0002,LU0003,LU0004
Cancer_Type,Breast,Prostate,Skin,Colon,Lung
Age,68,74,55,61,67
Gender,0,1,1,0,1
Smoking,7,8,7,6,10
Alcohol_Use,2,9,10,2,7
Obesity,8,8,7,2,4
Family_History,0,0,0,0,0
Diet_Red_Meat,5,0,3,6,6
Diet_Salted_Processed,3,3,3,2,3


In [5]:
X = df.drop(columns=['Risk_Level', 'Patient_ID', 'Cancer_Type'])

In [6]:
le = LabelEncoder()
y = le.fit_transform(df['Risk_Level'])

In [7]:
print(y)

[2 2 2 ... 1 2 2]


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Decision Tree

In [10]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_scaled, y_train)
y_pred_decision_tree = decision_tree.predict(X_test_scaled)

In [11]:
print("Decision Tree Classifier:")
print(classification_report(y_test, y_pred_decision_tree))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_decision_tree))

Decision Tree Classifier:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.98      1.00      0.99        65
           2       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       0.99      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix:
[[ 20   0   0]
 [  0  65   0]
 [  0   1 314]]


## Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression

logistic_regression = LogisticRegression(random_state=42, max_iter=100)
logistic_regression.fit(X_scaled, y_train)
y_pred_logistic_regression = logistic_regression.predict(X_test_scaled)

In [13]:
print("Logistic Regression:")
print(classification_report(y_test, y_pred_logistic_regression))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic_regression))

Logistic Regression:
              precision    recall  f1-score   support

           0       1.00      0.80      0.89        20
           1       0.97      0.95      0.96        65
           2       0.98      0.99      0.99       315

    accuracy                           0.98       400
   macro avg       0.98      0.92      0.95       400
weighted avg       0.98      0.98      0.98       400

Confusion Matrix:
[[ 16   0   4]
 [  0  62   3]
 [  0   2 313]]


## Remove some X variables

In [14]:
X_new = df.drop(columns=['Risk_Level', 'Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])
le = LabelEncoder()
y_new = le.fit_transform(df['Risk_Level'])

In [15]:
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42, stratify=y)

In [16]:
scaler = StandardScaler()
X_new_scaled = scaler.fit_transform(X_train_new)
X_test_new_scaled = scaler.transform(X_test_new)

## Decison Tree and Logistic Regression with new X

In [17]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_new_scaled, y_train_new)
y_pred_decision_tree_new = decision_tree.predict(X_test_new_scaled)
print("Decision Tree Classifier:")
print(classification_report(y_test_new, y_pred_decision_tree_new))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_decision_tree_new))

Decision Tree Classifier:
              precision    recall  f1-score   support

           0       0.28      0.35      0.31        20
           1       0.55      0.48      0.51        65
           2       0.85      0.86      0.86       315

    accuracy                           0.78       400
   macro avg       0.56      0.56      0.56       400
weighted avg       0.78      0.78      0.77       400

Confusion Matrix:
[[  7   0  13]
 [  0  31  34]
 [ 18  25 272]]


In [18]:
from sklearn.linear_model import LogisticRegression

logistic_regression = LogisticRegression(random_state=42, max_iter=100)
logistic_regression.fit(X_new_scaled, y_train_new)
y_pred_logistic_regression_new = logistic_regression.predict(X_test_new_scaled)
print("Logistic Regression:")
print(classification_report(y_test_new, y_pred_logistic_regression_new))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_logistic_regression_new))

Logistic Regression:
              precision    recall  f1-score   support

           0       0.50      0.20      0.29        20
           1       0.85      0.78      0.82        65
           2       0.91      0.96      0.93       315

    accuracy                           0.89       400
   macro avg       0.75      0.65      0.68       400
weighted avg       0.88      0.89      0.88       400

Confusion Matrix:
[[  4   0  16]
 [  0  51  14]
 [  4   9 302]]


## Smote - Synthetic Minority Over Sampling Technique

In [19]:
X_new = df.drop(columns=['Risk_Level', 'Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])
le = LabelEncoder()
y_new = le.fit_transform(df['Risk_Level'])
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42, stratify=y)

In [20]:
print(pd.Series(y_train_new).value_counts())

2    1259
1     259
0      82
Name: count, dtype: int64


In [21]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_new, y_train_new)

print(pd.Series(y_smote).value_counts())

2    1259
1    1259
0    1259
Name: count, dtype: int64


## Random Forest - Not Scaled

In [22]:
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_smote, y_smote)
y_pred_random_forest = random_forest.predict(X_test_new)
print("Random Forest Classifier:")
print(classification_report(y_test_new, y_pred_random_forest))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_random_forest))

Random Forest Classifier:
              precision    recall  f1-score   support

           0       0.41      0.35      0.38        20
           1       0.73      0.58      0.65        65
           2       0.88      0.92      0.90       315

    accuracy                           0.84       400
   macro avg       0.67      0.62      0.64       400
weighted avg       0.83      0.84      0.83       400

Confusion Matrix:
[[  7   0  13]
 [  0  38  27]
 [ 10  14 291]]


In [23]:
scaler = StandardScaler()
X_smote_scaled = scaler.fit_transform(X_smote)
X_test_new_scaled = scaler.transform(X_test_new)

## RandomForest - Scaled

In [24]:
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_smote_scaled, y_smote)
y_pred_random_forest = random_forest.predict(X_test_new_scaled)
print("Random Forest Classifier:")
print(classification_report(y_test_new, y_pred_random_forest))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_random_forest))

Random Forest Classifier:
              precision    recall  f1-score   support

           0       0.41      0.35      0.38        20
           1       0.73      0.58      0.65        65
           2       0.88      0.92      0.90       315

    accuracy                           0.84       400
   macro avg       0.67      0.62      0.64       400
weighted avg       0.83      0.84      0.83       400

Confusion Matrix:
[[  7   0  13]
 [  0  38  27]
 [ 10  14 291]]


## Optuna

### 📌 Tổng quan về Cơ chế Tối ưu hóa Siêu tham số với Optuna

1. **Optuna**: Thư viện tự động tìm kiếm siêu tham số thông minh thông qua việc tối ưu hóa một hàm mục tiêu (`objective`).
2. **TPESampler (Tree-structured Parzen Estimator)**: Bộ não thuật toán dựa trên phân phối xác suất Bayesian. Nó chia lịch sử thử nghiệm thành 2 nhóm: *tốt nhất* (high-performing) và *còn lại* (low-performing), từ đó tính toán gợi ý các tham số tiếp theo có xác suất cao nhất rơi vào nhóm tốt và tránh vùng điểm kém.
3. **Trial & Các phương thức Suggest**:
   - `Trial` đại diện cho một lượt thử nghiệm đơn lẻ.
   - `trial.suggest_int()` / `suggest_categorical()`: Các hàm yêu cầu bộ sampler gợi ý giá trị (số nguyên hoặc phân loại) phù hợp cho từng siêu tham số trong không gian tìm kiếm.
4. **StratifiedKFold**: Cơ chế chia dữ liệu K-Phần (K-Fold) giúp đảm bảo tỷ lệ giữa các phân lớp (classes) trong từng phần luôn đồng đều như tập dữ liệu gốc, cực kỳ quan trọng khi xử lý dữ liệu mất cân bằng nhằm đánh giá mô hình khách quan nhất.
5. **Macro F1-Score**: Chỉ số đánh giá bằng cách tính F1-Score riêng biệt cho từng lớp rồi lấy trung bình cộng giản đơn. Khác với F1 thông thường (chỉ dùng cho nhị phân) hay Accuracy/Micro F1 (dễ bị bias bởi lớp chiếm đa số), Macro F1 đánh giá công bằng hiệu năng của mô hình trên tất cả các lớp.

In [25]:
import optuna
from optuna.samplers import TPESampler #Tree-structured Parzen Estimator - "bộ não" quyết định cách chọn tham số tiếp theo.
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer #??
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib #save model

In [26]:
# Multiple class
# f1_score thông thường (mặc định là nhị phân - binary)
def macro_f1(y_true, y_pred):
    # average='macro' tính điểm f1_score cho từng class và lấy trung bình cộng thay vì dùng Accuracy
    # hay Micro f1 score chung kia (bị bias)
    return f1_score(y_true, y_pred, average='macro')

scorer = make_scorer(macro_f1, greater_is_better=True)

In [27]:
def objective(trial):
    # Định nghĩa dictionary chứa các hyperparameters cho RandomForestClassifier
    # trial.suggest_int('name', low, high, step)
    # trial.suggest_categorical('name', ['option1', 'option2'])
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss']),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
    }

    # Khởi tạo mô hình với các tham số từ params
    smote = SMOTE(random_state=42)
    scaler = StandardScaler()
    clf = RandomForestClassifier(**params, random_state=42)
    pl = Pipeline([('smote', smote), ('scaler', scaler), ('clf', clf)])

    # Đánh giá chéo (Cross-validation) sử dụng StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pl, X_train_new, y_train_new, cv=cv, scoring=scorer, n_jobs=-1)

    # scores = []
    # for train_idx, val_idx in cv.split(X_smote_scaled, y_smote):
    #   X_t, y_t = X_smote_scaled[train_idx], y_smote[train_idx]
    #   X_v, y_v = X_smote_scaled[val_idx], y_smote[val_idx]
    #   clf.fit(X_t, y_t)
    #   y_pred = clf.predict(X_v)
    #   scorer = macro_f1(y_v, y_pred)
    #   score.append(scorer)
    # return scores.mean()

    # Trả về điểm trung bình Macro F1
    return scores.mean()

### 📌 Vai trò của `Study` trong Optuna

`Study` là **cuộc thử nghiệm tổng thể** quản lý toàn bộ quá trình tối ưu hóa:
* **Quản lý mục tiêu:** Định nghĩa hướng tối ưu là cực đại hóa (`maximize`) hay cực tiểu hóa (`minimize`) thông qua hàm `objective`.
* **Lưu trữ lịch sử:** Lưu lại thông tin của tất cả các lượt chạy thử (`Trial`).
* **Tính năng nâng cao (khi dùng `storage` và `study_name`):**
  * **Khôi phục (Resume):** Chạy tiếp tục các lượt thử nghiệm cũ mà không phải làm lại từ đầu (`load_if_exists=True`).
  * **Tối ưu song song:** Cho phép nhiều máy/tiến trình cùng tham gia tối ưu một bài toán thông qua cơ sở dữ liệu chung (RDB).

```
# Khi bạn muốn lưu trữ vào SQLite để sau này tải lại hoặc vẽ đồ thị phân tích:
study = optuna.create_study(
    study_name='rf_macro_f1',
    direction='maximize',
    storage='sqlite:///optuna_studies.db', # Lưu vào file db
    load_if_exists=True # Nếu đã có tên này rồi thì chạy tiếp tục, không ghi đè
)
```

In [28]:
# Khởi tạo và chạy thử nghiệm Optuna
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42), study_name='rf_macro_f1')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best Trial:")
print("  Value (Macro F1):", study.best_trial.value)
print("  Params:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-10-05 09:31:28,551] A new study created in memory with name: rf_macro_f1


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-10-05 09:31:37,250] Trial 0 finished with value: 0.7071186512829282 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 6, 'criterion': 'gini', 'max_features': 'sqrt', 'class_weight': 'balanced_subsample', 'bootstrap': True}. Best is trial 0 with value: 0.7071186512829282.
[I 2026-10-05 09:31:45,420] Trial 1 finished with value: 0.6183413604495076 and parameters: {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 5, 'criterion': 'entropy', 'max_features': None, 'class_weight': 'balanced', 'bootstrap': True}. Best is trial 0 with value: 0.7071186512829282.
[I 2026-10-05 09:31:54,077] Trial 2 finished with value: 0.6533862217279165 and parameters: {'n_estimators': 200, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 10, 'criterion': 'gini', 'max_features': 'log2', 'class_weight': 'balanced_subsample', 'bootstrap': True}. Best is trial 0 with value: 0.7071186512829282.
[I 2026-10-05 09:32:05,

In [29]:
best_params = study.best_trial.params
best_params['random_state'] = 42
print(best_params)

{'n_estimators': 100, 'max_depth': 14, 'min_samples_split': 6, 'min_samples_leaf': 9, 'criterion': 'gini', 'max_features': 'sqrt', 'class_weight': 'balanced_subsample', 'bootstrap': True, 'random_state': 42}


In [30]:
random_forest = RandomForestClassifier(**best_params)
random_forest.fit(X_smote_scaled, y_smote)
y_pred_random_forest = random_forest.predict(X_test_new_scaled)
print("Random Forest Classifier:")
print(classification_report(y_test_new, y_pred_random_forest))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_random_forest))

Random Forest Classifier:
              precision    recall  f1-score   support

           0       0.33      0.45      0.38        20
           1       0.72      0.66      0.69        65
           2       0.89      0.89      0.89       315

    accuracy                           0.83       400
   macro avg       0.65      0.67      0.65       400
weighted avg       0.84      0.83      0.83       400

Confusion Matrix:
[[  9   0  11]
 [  0  43  22]
 [ 18  17 280]]


## XGBoost

In [31]:
from xgboost import XGBClassifier

In [32]:
X_new = df.drop(columns=['Risk_Level', 'Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])
y_new = df['Risk_Level']
le = LabelEncoder()
y_new = le.fit_transform(y)
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42, stratify=y)

In [33]:
smote = SMOTE(random_state=42)
xgb = XGBClassifier(use_label_encoder=True, eval_metrics='mlogloss', random_state=42, n_jobs=-1)
pl = Pipeline([('smote', smote), ('xgb', xgb)])

pl.fit(X_train_new, y_train_new)
y_pred_xgb = pl.predict(X_test_new)
print("XGBoost Classifier:")
print(classification_report(le.inverse_transform(y_test_new), le.inverse_transform(y_pred_xgb)))
print("Confusion Matrix:")
print(confusion_matrix(le.inverse_transform(y_test_new), le.inverse_transform(y_pred_xgb)))

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:35:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Classifier:
              precision    recall  f1-score   support

           0       0.35      0.40      0.37        20
           1       0.77      0.71      0.74        65
           2       0.90      0.91      0.91       315

    accuracy                           0.85       400
   macro avg       0.67      0.67      0.67       400
weighted avg       0.85      0.85      0.85       400

Confusion Matrix:
[[  8   0  12]
 [  0  46  19]
 [ 15  14 286]]


## XGBoost + Optuna

In [34]:
import optuna
from optuna.samplers import TPESampler #Tree-structured Parzen Estimator - "bộ não" quyết định cách chọn tham số tiếp theo.
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, make_scorer #??
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib #save model

In [35]:
# Settings
TARGET_LABEL = 'High'
LABEL_ORDER = ['High', 'Medium', 'Low']
N_TRIALS = 50
CV_FOLDS = 5

In [36]:
X_new = df.drop(columns=['Risk_Level', 'Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])
y_new = df['Risk_Level']
le = LabelEncoder()
y_new = le.fit_transform(df.loc[X_new.index, 'Risk_Level'])
print(le.classes_)
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42, stratify=y)

['High' 'Low' 'Medium']


In [37]:
# Lấy index dạng số của class mục tiêu 'High' trong le.classes_
target_index = np.where(le.classes_ == TARGET_LABEL)[0][0]
print(f'Target index: {target_index}')
print(f'Target label: {le.classes_}')

Target index: 0
Target label: ['High' 'Low' 'Medium']


In [38]:
# Lay value recall moi thang High-Risk
def macro_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, average=None, labels=list(range(len(le.classes_))))[target_index]

scorer_recall = make_scorer(macro_recall, greater_is_better=True)

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'mlogloss',
        'objective': 'multi:softmax',
        'use_label_encoder': False,
        'num_class': len(LABEL_ORDER)
    }

    smote = SMOTE(random_state=42)
    scaler = StandardScaler()
    clf = XGBClassifier(**params)

    pl = Pipeline([('smote', smote), ('scaler', scaler), ('clf', clf)])

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    scores = cross_val_score(pl, X_train_new, y_train_new, cv=cv, scoring=scorer_recall, n_jobs=-1)

    return scores.mean()

In [41]:
study = optuna.create_study(study_name='xgb_macro_recall', direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best Trial:")
print("  Value (Macro Recall):", study.best_trial.value)
print("  Params:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-10-05 09:44:09,329] A new study created in memory with name: xgb_macro_recall


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-10-05 09:44:16,497] Trial 0 finished with value: 0.7071186512829282 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 6, 'criterion': 'gini', 'max_features': 'sqrt', 'class_weight': 'balanced_subsample', 'bootstrap': True}. Best is trial 0 with value: 0.7071186512829282.
[I 2026-10-05 09:44:25,374] Trial 1 finished with value: 0.6183413604495076 and parameters: {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 5, 'criterion': 'entropy', 'max_features': None, 'class_weight': 'balanced', 'bootstrap': True}. Best is trial 0 with value: 0.7071186512829282.
[I 2026-10-05 09:44:33,164] Trial 2 finished with value: 0.6533862217279165 and parameters: {'n_estimators': 200, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 10, 'criterion': 'gini', 'max_features': 'log2', 'class_weight': 'balanced_subsample', 'bootstrap': True}. Best is trial 0 with value: 0.7071186512829282.
[I 2026-10-05 09:44:40,

In [42]:
best_params = study.best_trial.params
best_params['random_state'] = 42

smote = SMOTE(random_state=42)
scaler = StandardScaler()
xgb = XGBClassifier(**best_params)
pipe = Pipeline([('smote', smote), ('scaler', scaler), ('xgb', xgb)])

pipe.fit(X_train_new, y_train_new)
y_pred_xgb = pipe.predict(X_test_new)
print("XGBoost Classifier:")
print(classification_report(le.inverse_transform(y_test_new), le.inverse_transform(y_pred_xgb)))
print("Confusion Matrix:")
print(confusion_matrix(le.inverse_transform(y_test_new), le.inverse_transform(y_pred_xgb)))

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:47:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "bootstrap", "class_weight", "criterion", "max_features", "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Classifier:
              precision    recall  f1-score   support

        High       0.43      0.45      0.44        20
         Low       0.75      0.66      0.70        65
      Medium       0.90      0.92      0.91       315

    accuracy                           0.85       400
   macro avg       0.69      0.68      0.68       400
weighted avg       0.85      0.85      0.85       400

Confusion Matrix:
[[  9   0  11]
 [  0  43  22]
 [ 12  14 289]]


## XGBoost + Class-Weight

In [43]:
X_new = df.drop(columns=['Risk_Level', 'Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])
y_new = df['Risk_Level']
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.2, random_state=42, stratify=y)

In [46]:
le = LabelEncoder()
y_train_new = le.fit_transform(df.loc[X_train_new.index, 'Risk_Level'])
y_test_new = le.transform(df.loc[X_test_new.index, 'Risk_Level'])
print(le.classes_)
classes, counts = np.unique(y_train_new, return_counts=True)
print(classes, counts)
classes_weight = {cl: max(counts)/count for cl, count in zip(classes, counts)}
print(classes_weight)
sample_weight = np.array([classes_weight[cl] for cl in y_train_new])
print(sample_weight)

['High' 'Low' 'Medium']
[0 1 2] [  82  259 1259]
{np.int64(0): np.float64(15.353658536585366), np.int64(1): np.float64(4.861003861003861), np.int64(2): np.float64(1.0)}
[1. 1. 1. ... 1. 1. 1.]


In [47]:
xgb_weighted = XGBClassifier(
    objective='multi:softmax',
    num_class=len(classes),
    eval_metric='mlogloss',
    random_state=42,
    n_estimators=105,
    max_depth=3,
    learning_rate=0.014843806162322944,
    subsample=0.7730982444721965,
    colsample_bytree=0.8509237341976973,
    gamma=4.134336554829186,
    reg_alpha=4.453508368364819,
    reg_lambda=2.616923339470738,
    n_jobs=-1
)

pipe = Pipeline([('xgb', xgb_weighted)])

pipe.fit(X_train_new, y_train_new, xgb__sample_weight=sample_weight)
y_pred_xgb = pipe.predict(X_test_new)

print("XGBoost Classifier:")
print(classification_report(y_test_new, y_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_xgb))

XGBoost Classifier:
              precision    recall  f1-score   support

           0       0.21      0.75      0.33        20
           1       0.43      0.80      0.56        65
           2       0.91      0.60      0.72       315

    accuracy                           0.64       400
   macro avg       0.52      0.72      0.54       400
weighted avg       0.80      0.64      0.68       400

Confusion Matrix:
[[ 15   0   5]
 [  0  52  13]
 [ 56  70 189]]


## XGBoost + Class-Weight + Optuna

In [65]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

scorer = make_scorer(macro_f1, greater_is_better=True)

def objective_xgb_weighted(trial):
    # 1. Định nghĩa các siêu tham số cần tối ưu cho XGBoost
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'use_label_encoder': False,
        'eval_metric': 'mlogloss',
    }

    xgb_weighted = XGBClassifier(**params)
    scores = []

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    for train_idx, val_idx in cv.split(X_train_new, y_train_new):
      X_t, y_t = X_train_new.iloc[train_idx], y_train_new[train_idx]
      X_v, y_v = X_train_new.iloc[val_idx], y_train_new[val_idx]

      # class weight cho cv se khac cho toan bo nen he so se khac
      classes, counts = np.unique(y_t, return_counts=True)
      classes_weight = {cl: max(counts)/count for cl, count in zip(classes, counts)}
      sample_weight = np.array([classes_weight[cl] for cl in y_t])

      xgb_weighted.fit(X_t, y_t, sample_weight=sample_weight)
      y_pred = xgb_weighted.predict(X_v)
      scorer = macro_f1(y_v, y_pred)
      scores.append(scorer)

    return np.mean(scores)

In [66]:
study = optuna.create_study(study_name='xgb_macro_f1', direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective_xgb_weighted, n_trials=40, show_progress_bar=True)

print("Best Trial:")
print("  Value (Macro Recall):", study.best_trial.value)
print("  Params:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-10-05 10:27:15,429] A new study created in memory with name: xgb_macro_f1


  0%|          | 0/40 [00:00<?, ?it/s]

/tmp/ipykernel_1796/3389127331.py:11: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:27:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:27:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:27:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, it

[I 2026-10-05 10:27:19,015] Trial 0 finished with value: 0.6842784646293902 and parameters: {'n_estimators': 181, 'max_depth': 12, 'learning_rate': 0.06504856968981275, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.4936111842654619, 'gamma': 0.7799726016810132, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.6842784646293902.


/tmp/ipykernel_1796/3389127331.py:11: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:27:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:27:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:27:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, it

...

[I 2026-10-05 10:28:26,255] Trial 37 finished with value: 0.7205396103027951 and parameters: {'n_estimators': 275, 'max_depth': 12, 'learning_rate': 0.11551355660699468, 'subsample': 0.6717727101598435, 'colsample_bytree': 0.4248881310906546, 'gamma': 1.310371427647964, 'reg_alpha': 2.194872705717262, 'reg_lambda': 1.09217570455677}. Best is trial 35 with value: 0.7313066455210327.


/tmp/ipykernel_1796/3389127331.py:11: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:28:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:28:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:28:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, it

[I 2026-10-05 10:28:27,266] Trial 38 finished with value: 0.7074674368568588 and parameters: {'n_estimators': 119, 'max_depth': 9, 'learning_rate': 0.07721019620285259, 'subsample': 0.611047478900963, 'colsample_bytree': 0.49047359060525697, 'gamma': 1.5951071315856904, 'reg_alpha': 1.6661099328607603, 'reg_lambda': 1.7169257605456993}. Best is trial 35 with value: 0.7313066455210327.


/tmp/ipykernel_1796/3389127331.py:11: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:28:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:28:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [10:28:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, it

[I 2026-10-05 10:28:28,742] Trial 39 finished with value: 0.7252631878967999 and parameters: {'n_estimators': 269, 'max_depth': 11, 'learning_rate': 0.08583602433873457, 'subsample': 0.6975089649342218, 'colsample_bytree': 0.4723456689275969, 'gamma': 1.6933013090171245, 'reg_alpha': 0.9309144561618029, 'reg_lambda': 0.6826373863220995}. Best is trial 35 with value: 0.7313066455210327.
Best Trial:
  Value (Macro Recall): 0.7313066455210327
  Params:
    n_estimators: 260
    max_depth: 11
    learning_rate: 0.06590577336432804
    subsample: 0.6514465904887482
    colsample_bytree: 0.4827918762525874
    gamma: 1.498462222464858
    reg_alpha: 0.9826959320211304
    reg_lambda: 1.5819352328509684


In [67]:
best_params = study.best_trial.params
best_params['random_state'] = 42
best_params['objective'] = 'multi:softmax'
best_params['num_class'] = len(np.unique(y_train_new))
best_params['eval_metric'] = 'mlogloss'

xgb_weighted = XGBClassifier(**best_params)

xgb_weighted.fit(X_train_new, y_train_new, sample_weight=sample_weight)
y_pred_xgb = xgb_weighted.predict(X_test_new)

print("XGBoost Classifier:")
print(classification_report(y_test_new, y_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test_new, y_pred_xgb))

XGBoost Classifier:
              precision    recall  f1-score   support

           0       0.50      0.45      0.47        20
           1       0.75      0.83      0.79        65
           2       0.93      0.91      0.92       315

    accuracy                           0.88       400
   macro avg       0.73      0.73      0.73       400
weighted avg       0.88      0.88      0.88       400

Confusion Matrix:
[[  9   0  11]
 [  0  54  11]
 [  9  18 288]]


In [68]:
joblib.dump(xgb_weighted, 'xgb_weighted.pkl')
print("Model save successfully")

Model save successfully


In [69]:
import joblib
artifacts = {
    'model': xgb_weighted,
    'scaler': scaler,
    'label_encoder': le
}

joblib.dump(artifacts, '../models/xgb_complete_pipeline.pkl')
print("Đã lưu thành công model, scaler và label_encoder vào file 'xgb_complete_pipeline.pkl'!")

Đã lưu thành công model, scaler và label_encoder vào file 'xgb_complete_pipeline.pkl'!
